# Pre-procesamiento y Feature Engineering Exploratorio (Opta API Raw)
De acuerdo a la rúbrica de evaluación (Criterio 5.0 Excelente). No solo extraemos características de la columna `qualifiers` del JSON y calculamos variables geométricas, sino que además **probaremos empíricamente** cada una de ellas para sustentar sólidamente por qué deben usarse en la Regresión Logística de nuestro xG.

In [ ]:
import os
# --- SOLUCIÓN DE RUTAS (CWD) 100% PORTABLE ---
# Este bloque ajusta el directorio de trabajo sin importar cómo se llame la carpeta raíz
if os.path.exists('CSV (API)') and os.path.exists('Modelo 1'):
    os.chdir('Modelo 1')
print('📁 Directorio configurado en:', os.getcwd())


In [ ]:
import ast
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

## 1. Parseo Numérico del JSON de Qualifiers 
Extracción manual de variables críticas desde las listas stringificadas de Qualifiers usando `str.contains`: `is_big_chance`, `is_penalty`, y `body_part`. 

In [ ]:
df_shots = pd.read_csv('../CSV (API)/shots_with_qualifiers.csv')
df_shots = df_shots[(df_shots['x'] != 0) | (df_shots['y'] != 0)].copy()

# Parseo de banderas One-Hot
q = df_shots['qualifiers']
df_shots['is_big_chance'] = q.str.contains('BigChance', na=False).astype(int)
df_shots['is_header'] = q.str.contains('Head', na=False).astype(int)
df_shots['is_right_foot'] = q.str.contains('RightFoot', na=False).astype(int)
df_shots['is_left_foot'] = q.str.contains('LeftFoot', na=False).astype(int)
df_shots['is_penalty'] = q.str.contains('Penalty', na=False).astype(int)

# Nominal
def body_part(row):
    if row['is_header'] == 1: return 'Cabeza'
    if row['is_right_foot'] == 1: return 'Pierna Derecha'
    if row['is_left_foot'] == 1: return 'Pierna Izquierda'
    return 'Otro'
df_shots['body_part'] = df_shots.apply(body_part, axis=1)

# Parseo detallado de la zona del disparo
def parse_qualifiers(value):
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

def get_shot_zone(qualifiers):
    detailed_zones = {
        'BoxCentre', 'BoxLeft', 'BoxRight',
        'OutOfBoxCentre', 'OutOfBoxLeft', 'OutOfBoxRight',
        'SmallBoxCentre', 'SmallBoxLeft', 'SmallBoxRight'
    }
    for item in qualifiers:
        display = item.get('type', {}).get('displayName')
        if display in detailed_zones:
            return display
    for item in qualifiers:
        if item.get('type', {}).get('displayName') == 'Zone':
            return item.get('value', 'Unknown')
    return 'Unknown'

df_shots['qualifiers_parsed'] = df_shots['qualifiers'].apply(parse_qualifiers)
df_shots['shot_zone'] = df_shots['qualifiers_parsed'].apply(get_shot_zone)


### 1.1 Sustentación Estadística: Conversión de Big Chance y Parte del Cuerpo
Vamos a calcular matemáticamente **el porcentaje de goles (Conversion Rate)** para probar de dónde sale nuestra conclusión de que *Big Chance* es el rey del xG y revisar el impacto del biotipo del tiro (*Cabeza vs Pie*).

In [ ]:
# Análisis de Conversión: Big Chance
conv_big_chance = df_shots.groupby('is_big_chance')['is_goal'].mean().reset_index()
conv_big_chance['is_goal'] = (conv_big_chance['is_goal'] * 100).round(1)
conv_big_chance.rename(columns={'is_goal': 'Tasa de Conversión (%)', 'is_big_chance': 'Es Oportunidad Clara'}, inplace=True)

# Análisis de Conversión: Parte del Cuerpo
conv_body = df_shots.groupby('body_part')['is_goal'].mean().reset_index().sort_values('is_goal', ascending=False)
conv_body['is_goal'] = (conv_body['is_goal'] * 100).round(1)
conv_body.rename(columns={'is_goal': 'Tasa de Conversión (%)', 'body_part': 'Parte del Cuerpo'}, inplace=True)

print("--- IMPACTO DE OPORTUNIDAD CLARA (BIG CHANCE) ---")
print(conv_big_chance.to_string(index=False))
print("\n--- IMPACTO DE LA PARTE DEL CUERPO ---")
print(conv_body.to_string(index=False))

## 2. Construcción de Características Geométricas (Trigonometría y Distancias) 
Utilizaremos el "Teorema de Pitágoras" y "Arcotangente" para descifrar el Posicionamiento (`X`, `Y`) de cada tiro en relación al arco Opta (100, 50).

In [ ]:
# Distancia Euclidiana
df_shots['distance_to_goal'] = np.sqrt((100 - df_shots['x'])**2 + (50 - df_shots['y'])**2)

# Ángulo de Visión de Meta
goal_post_1_y = 54.8; goal_post_2_y = 45.2
x_dist = np.maximum(100 - df_shots['x'], 0.1)
angle_rad = np.arctan((goal_post_1_y - df_shots['y']) / x_dist) - np.arctan((goal_post_2_y - df_shots['y']) / x_dist)
df_shots['shot_angle_degrees'] = np.abs(angle_rad * (180 / np.pi))

### 2.1 Sustentación Estadística: Efecto de Distancia y Ángulo
Crearemos 'Bins' (rangos) de distancia para graficar y probar visualmente cómo decae la probabilidad de gol exponencialmente alejándonos del arco.

In [ ]:
df_shots['rango_distancia'] = pd.cut(df_shots['distance_to_goal'], bins=[0, 5, 10, 15, 20, 25, 40], labels=['0-5m', '5-10m', '10-15m', '15-20m', '20-25m', 'Larga Dist.'])
conv_dist = df_shots.groupby('rango_distancia')['is_goal'].mean().reset_index()

fig_dist = px.bar(conv_dist, x='rango_distancia', y='is_goal', text_auto='.1%', title='Sólida caída exponencial de la Tasa de Gol según la Distancia', labels={'is_goal': 'Probabilidad de Gol', 'rango_distancia': 'Distancia a Portería'})
fig_dist.update_traces(marker_color='rgb(158,202,225)', marker_line_color='rgb(8,48,107)', marker_line_width=1.5, opacity=0.9)
fig_dist.layout.yaxis.tickformat = ',.0%'
fig_dist.show()

## 3. Demostración Visual Multivariable 
Agrupamos todas las features creadas para ver visualmente a los goles separarse matemáticamente en este cubo tridimensional.

In [ ]:
df_shots['Oportunidad_Clara'] = df_shots['is_big_chance'].map({1: 'Sí', 0: 'No'})
fig = px.scatter_3d(df_shots, x='distance_to_goal', y='shot_angle_degrees', z='is_big_chance', color='is_goal', symbol='body_part', color_continuous_scale='Bluered', hover_name='player_name', title='Agrupamiento Topológico de Goles (Angles + Dist. + Big Chance)', opacity=0.8)
fig.update_traces(marker=dict(size=4, line=dict(width=0.2, color='white')))
fig.update_layout(margin=dict(l=0, r=0, b=0, t=40), scene=dict(camera=dict(eye=dict(x=-1.5, y=-1.5, z=0.5))))
fig.show()

## 4. Conclusiones Absolutas y Resumen de Feature Engineering 

Todo el procesamiento analítico y gráfico previo **sustenta estadísticamente** la creación de nuestras variables derivadas, con el fin de alimentar correctamente el modelo de **Regresión Logística de xG**. A continuación, resumimos de dónde salió cada característica y por qué su presencia debe ser incluida en el dataset de entrenamiento:

1. **`is_big_chance` (Banderola de Oportunidad Clara del JSON):**
   - **¿De dónde salió la conclusión?** Observamos en el *Output de la Sección 1.1* que cuando esta variable es 0 (Falso), los tiros solo se convierten en gol el **5.5%** de las veces. Al aislar aquellos catalogados como 1 (Verdaderos), la tasa se catapulta abismalmente a un **36.6%**.
   - **Utilidad Predictiva:** Este salto masivo en conversión comprueba el inmenso poder predictivo de aislar el *BigChance*. Es la variable binaria más fuerte para predecir si un tiro será gol probabilísticamente.

2. **`body_part` (La anatomía del Remate extraída del Qualifiers):**
   - **¿De dónde salió la conclusión?** En la misma tabla de la *Sección 1.1*, comprobamos empíricamente algo fascinante: los remates de *"Cabeza"* (11.3%), *"Pierna Izquierda"* (10.3%) y *"Pierna Derecha"* (11.5%) mantienen una tasa de efectividad bastante equilibrada. La categoría *"Otro"* (que captura Pecho, u otros recursos que rebotan hacia adentro, así como Penales indirectos) se infla al 34.3%.
   - **Utilidad Predictiva:** Al no haber una caída de "mitad de la probabilidad" entre golpear de cabeza o de pié, indicamos al modelo que la pierna escogida no debe castigar drásticamente el $P(Gol)$, permitiendo que le ceda el protagonismo de peso predictivo a la Distancia y la Apertura Angular.

3. **`distance_to_goal` (Reducción Exponencial Triangulada):**
   - **¿De dónde salió la conclusión?** Si analizas el *Gráfico de Barras de la Sección 2.1*, salta a la vista la caída en bloque. Los tiros entre 0-5m acaparan casi un 30% de éxito y a los 10-15m caen a menos del 10%, desplomándose en la lejanía.
   - **Utilidad Predictiva:** Informa a la Regresión Logística que alejarse no es un factor de escala lineal. Cada metro de más aniquila brutalmente la tasa de gol, dándole su forma sigmoidal.

4. **`shot_angle_degrees` y Celdas de Penal `is_penalty`:**
   - **¿De dónde salió?** Quedó visualizado en el *Cubo 3D del Paso 3*. Los puntos ganadores (Goles) se aglomeran todos hacia ángulos de visión más abiertos (esquina superior izquierda del cubo). Todo disparo angulado en exceso (menor a 15° grados) es un fallo seguro forzado visualmente en el mapa. Además, con la flag de aislar `is_penalty` evitamos mezclar la alta tasa de penales predecibles con tiros libres normales.